# Assignment 3

## Part 1

In this part, you will work with data on New York Times Hardcover Fiction Bestsellers, 1931-2020 period. The data was curated by the R for Data Science team and published [here](https://github.com/rfordatascience/tidytuesday/blob/main/data/2022/2022-05-10/readme.md). There are two datasets: nyt_full.tsv and nyt_titles.tsv. We will only work with the former. Here is a detailed description of the data (please read carefully): https://web.archive.org/web/20230307140717/https://data.post45.org/wp-content/uploads/2022/01/NYT-Data-Description.pdf


There may be some mistakes in the data, such as the same author appearing with different spellings (for example, once with an initial and once without one). Do not try to correct them. Just take the data as it is. If the spelling is different, then the person is different. For determining the uniqueness of a book, use `title_id`.

The code chunk below loads the dataset for you and stores into a pandas DataFrame object called `nyt_full`. Please run the code chunk before proceeding to the exercises.

In [19]:
import pandas as pd
nyt_full = pd.read_csv("https://raw.githubusercontent.com/rfordatascience/tidytuesday/refs/heads/main/data/2022/2022-05-10/nyt_full.tsv", sep = "\t")
nyt_full

,year,week,rank,title_id,title,author
0,1931,1931-10-12,1,6477,THE TEN COMMANDMENTS,Warwick Deeping
1,1931,1931-10-12,2,1808,FINCHE'S FORTUNE,Mazo de la Roche
2,1931,1931-10-12,3,5304,THE GOOD EARTH,Pearl S. Buck
3,1931,1931-10-12,4,4038,SHADOWS ON THE ROCK,Willa Cather
4,1931,1931-10-12,5,3946,SCARMOUCHE THE KING MAKER,Rafael Sabatini
...,...,...,...,...,...,...
60381,2020,2020-12-06,11,2332,I WOULD LEAVE ME IF I COULD,Halsey
60382,2020,2020-12-06,12,6601,THE VANISHING HALF,Brit Bennett
60383,2020,2020-12-06,13,7239,WHERE THE CRAWDADS SING,Delia Owens
60384,2020,2020-12-06,14,482,ANXIOUS PEOPLE,Fredrik Backman


## Question 1 (2.5 points)

Find the book that was at the top of the list (i.e. the book's rank was 1) for the most number of weeks. Print the following:

- name of the book
- name of the author
- the number of weeks it was at the top

(Hint: If you have a list or a similar iterable object---such as a column from a pandas.DataFrame whose unique values you want to count---you can use [counter class from collections](https://docs.python.org/3/library/collections.html#collections.Counter). Then, you can use the .most_common() method to get the most commonly occuring element. This is all doable without these too, though).

In [20]:
from collections import Counter

def book_with_most_top_weeks(df):
    top_books = df[df['rank'] == 1]
    book_counts = Counter(top_books['title_id'])
    most_common_book = book_counts.most_common(1)[0]
    book_title = top_books[top_books['title_id'] == most_common_book[0]]['title'].iloc[0]
    book_author = top_books[top_books['title_id'] == most_common_book[0]]['author'].iloc[0]
    weeks_at_top = most_common_book[1]
    return book_title, book_author, weeks_at_top

title, author, weeks = book_with_most_top_weeks(nyt_full)
print(f"Book: {title}, Author: {author}, Weeks at Top: {weeks}")

Book: THE DA VINCI CODE, Author: Dan Brown, Weeks at Top: 59


## Question 2 (2.5 points)

Find the author who has had the most number of unique books at the top of the list. Print the following:

- name of the author
- how many books they had at the top


In [21]:
def author_with_most_unique_top_books(df):
    top_books = df[df['rank'] == 1]
    unique_books_by_author = top_books.groupby('author')['title_id'].nunique()
    most_prolific_author = unique_books_by_author.idxmax()
    unique_books_count = unique_books_by_author.max()
    return most_prolific_author, unique_books_count

author, count = author_with_most_unique_top_books(nyt_full)
print(f"Author: {author}, Unique Books at Top: {count}")

Author: Stephen King, Unique Books at Top: 40


## Part 2

In this assignment, you will work with data from New York City's Taxi and Limouisine Commission. Specifically, we'll look at data about Yellow Taxi Trip records from the first two months of 2024, i.e. January and February. The data comes from the links on this website:

https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page

For your convenience, the next code chunk loads the data as two separate pandas dataframe objects.

In [22]:
import pandas as pd
jan_trips = pd.read_parquet("https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet")
feb_trips = pd.read_parquet("https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-02.parquet")
mar_trips = pd.read_parquet("https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-03.parquet")

Each row in the dataset stands for a single trip. [This document](https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf) from their website explains what each variable in these dataframes means.

There are some mistakes in the data (it is as provided by NYC Gov). There are some rows where the date is not in January or February 2024. Find them and remove them before doing any analysis.

Another data we will use pertains to weather in New York City in this period and comes from the NOAA (National Oceanic and Atmospheric Administration). The following chunk loads that data into a dataframe called `weather`.

In [23]:
weather = pd.read_csv("https://raw.githubusercontent.com/omerfyalcin/colab-data/refs/heads/main/3816901.csv")

The `weather` dataframe contains temperature, precipitation, and similar measurements from various stations in the vicinity of New York City. It has the following columns:

- STATION - A unique identifier for the station
- NAME - The station name
- DATE - date
- DAPR - Number of days included in the multiday precipitation total (MDPR)
- MDPR - Multiday precipitation total (use with DAPR and DWPR, if available)
- PRCP - Precipitation
- TMAX - Maximum temperature
- TAVG - Average Temperature.
- TMIN - Minimum temperature
- TOBS - Temperature at the time of observation
- SNOW - Snowfall
- SNWD - Snow depth

For our purposes, we will use the measurements from the Central Park station as representative of all of New York City. That station has "CENTRAL PARK" in its name; no other station has it.


Answer the following questions using these data. Put your code / text for every question below that question. (Add text chunks if/when needed).

## Question 3 (1 point)

What is the total number of trips in each month (January, February, March)? Print the result.

In [24]:
taxi_data = pd.concat([jan_trips, feb_trips, mar_trips])
taxi_data['tpep_pickup_datetime'] = pd.to_datetime(taxi_data['tpep_pickup_datetime'])
taxi_data['pickup_date'] = taxi_data['tpep_pickup_datetime'].dt.date
taxi_data['pickup_month'] = pd.to_datetime(taxi_data['pickup_date']).dt.month


filtered_taxi_data = taxi_data[
    (taxi_data['pickup_date'] >= pd.Timestamp("2024-01-01").date()) &
    (taxi_data['pickup_date'] <= pd.Timestamp("2024-03-31").date())
]


total_trips_per_month = filtered_taxi_data['pickup_month'].value_counts().sort_index()


for month, trips in total_trips_per_month.items():
    print(f"Month {month}: {trips} trips")

Month 1: 2964617 trips
Month 2: 3007533 trips
Month 3: 3582607 trips


## Question 4 (1 point)

On what day was the average trip distance shortest? On what day was it longest? Print the result. (If a trip starts on one day and ends on another, assume that the trip belongs to the day on which it started.)

In [25]:
def shortest_longest_trip_days(df):
    avg_trip_distance_by_day = df.groupby('pickup_date')['trip_distance'].mean()
    shortest_day = avg_trip_distance_by_day.idxmin()
    longest_day = avg_trip_distance_by_day.idxmax()
    return shortest_day, longest_day

shortest_day, longest_day = shortest_longest_trip_days(taxi_data)
print(f"Shortest Trip Day: {shortest_day}, Longest Trip Day: {longest_day}")

Shortest Trip Day: 2008-12-31, Longest Trip Day: 2024-03-25


## Question 5 (1.5 points)

Is the mean trip distance positively or negatively correlated with precipitation? Answer by finding and printing the Pearson's correlation coefficient between mean trip distance and daily precipitation. (https://numpy.org/doc/2.1/reference/generated/numpy.corrcoef.html)

In [26]:
def correlation_trip_distance_precipitation(taxi_df, weather_df):
    avg_trip_distance = taxi_df.groupby('pickup_date')['trip_distance'].mean().reset_index()
    weather_df['DATE'] = pd.to_datetime(weather_df['DATE']).dt.date
    merged_df = avg_trip_distance.merge(
        weather_df[['DATE', 'PRCP']], left_on='pickup_date', right_on='DATE', how='inner'
    )
    correlation = np.corrcoef(merged_df['trip_distance'], merged_df['PRCP'])[0, 1]
    return correlation

precip_correlation = correlation_trip_distance_precipitation(taxi_data, central_park_weather)
print(f"Correlation between trip distance and precipitation: {precip_correlation}")

Correlation between trip distance and precipitation: -0.004066744887184293


## Question 6 (1.5 points)

What was the mean tip amount on days where the minimum temperature was greater than or equal to 35?

What was the mean tip amount on days where the minimum temperature was less than 35?

In [27]:
def mean_tip_based_on_temp(taxi_df, weather_df):
    weather_df['DATE'] = pd.to_datetime(weather_df['DATE']).dt.date
    merged_df = taxi_df.merge(
        weather_df[['DATE', 'TMIN']], left_on='pickup_date', right_on='DATE', how='inner'
    )
    high_temp_tips = merged_df[merged_df['TMIN'] >= 35]['tip_amount'].mean()
    low_temp_tips = merged_df[merged_df['TMIN'] < 35]['tip_amount'].mean()
    return high_temp_tips, low_temp_tips

high_temp_mean_tip, low_temp_mean_tip = mean_tip_based_on_temp(taxi_data, central_park_weather)
print(f"Mean Tip (TMIN >= 35): {high_temp_mean_tip}, Mean Tip (TMIN < 35): {low_temp_mean_tip}")

Mean Tip (TMIN >= 35): 3.2586244970092197, Mean Tip (TMIN < 35): 3.2899261724035878
